# Processing Notebook

Notebook nay tu chua toan bo logic preprocessing.
Ban co the chay preprocessing tu raw CSV ngay trong notebook ma khong can phu thuoc vao module Python rieng.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import logging
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("preprocess_notebook")


def resolve_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return cwd


REPO_ROOT = resolve_repo_root()
print("Repo root :", REPO_ROOT)


In [ ]:
# Preprocessing constants
FILES = [
    "data/raw/T_ONTIME_REPORTING_2021.csv",
    "data/raw/T_ONTIME_REPORTING_2022.csv",
    "data/raw/T_ONTIME_REPORTING_2023.csv",
    "data/raw/T_ONTIME_REPORTING_2024.csv",
    "data/raw/T_ONTIME_REPORTING_2025.csv",
]

CHUNKSIZE = 300_000
TARGET = "ARR_DEL15"
TRAIN_YEARS = {2021, 2022, 2023, 2024}
TEST_YEARS = {2025}

CANCEL_NULL_COLS = [
    "DEP_TIME", "ARR_TIME", "WHEELS_OFF", "WHEELS_ON", "TAXI_OUT", "TAXI_IN",
    "ACTUAL_ELAPSED_TIME", "AIR_TIME",
    "DEP_DELAY", "DEP_DELAY_NEW", "DEP_DEL15", "DEP_DELAY_GROUP",
    "ARR_DELAY", "ARR_DELAY_NEW", "ARR_DEL15", "ARR_DELAY_GROUP",
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
    "FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME",
]
DIVERT_NULL_COLS = [
    "ARR_TIME", "ARR_DELAY", "ARR_DELAY_NEW", "ARR_DEL15", "ARR_DELAY_GROUP", "ARR_TIME_BLK",
    "WHEELS_ON", "TAXI_IN",
]
HHMM_COLS = [
    "CRS_DEP_TIME", "DEP_TIME", "WHEELS_OFF", "WHEELS_ON",
    "CRS_ARR_TIME", "ARR_TIME", "FIRST_DEP_TIME",
]
DELAY_CAUSE_COLS = [
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
]
FREQ_ENCODE_COLS = ["OP_CARRIER", "ORIGIN", "DEST", "ROUTE", "DEP_TIME_BLK"]
OTP_GROUP_COLS = ["ORIGIN", "OP_CARRIER"]
DUPLICATE_KEY_COLS = ["YEAR", "DAY_OF_MONTH", "OP_CARRIER", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST", "CRS_DEP_TIME"]

DTYPE_MAP_INT = {
    "YEAR": "Int16", "DAY_OF_MONTH": "Int16", "DAY_OF_WEEK": "Int16",
    "OP_CARRIER_AIRLINE_ID": "Int32", "OP_CARRIER_FL_NUM": "Int32",
    "ORIGIN_AIRPORT_ID": "Int32", "ORIGIN_AIRPORT_SEQ_ID": "Int32",
    "ORIGIN_CITY_MARKET_ID": "Int32", "ORIGIN_STATE_FIPS": "Int16", "ORIGIN_WAC": "Int16",
    "DEST_AIRPORT_ID": "Int32", "DEST_AIRPORT_SEQ_ID": "Int32",
    "DEST_CITY_MARKET_ID": "Int32", "DEST_STATE_FIPS": "Int16", "DEST_WAC": "Int16",
    "DEP_DEL15": "Int16", "DEP_DELAY_GROUP": "Int16",
    "ARR_DEL15": "Int16", "ARR_DELAY_GROUP": "Int16",
    "CANCELLED": "Int16", "DIVERTED": "Int16",
    "DISTANCE_GROUP": "Int16", "FLIGHTS": "Int16",
    "DIV_AIRPORT_LANDINGS": "Int16", "DIV_REACHED_DEST": "Int16",
}
DTYPE_MAP_FLOAT = {
    "DEP_DELAY": "float32", "DEP_DELAY_NEW": "float32",
    "ARR_DELAY": "float32", "ARR_DELAY_NEW": "float32",
    "TAXI_OUT": "float32", "TAXI_IN": "float32",
    "CRS_ELAPSED_TIME": "float32", "ACTUAL_ELAPSED_TIME": "float32",
    "AIR_TIME": "float32", "DISTANCE": "float32",
    "CARRIER_DELAY": "float32", "WEATHER_DELAY": "float32",
    "NAS_DELAY": "float32", "SECURITY_DELAY": "float32", "LATE_AIRCRAFT_DELAY": "float32",
    "TOTAL_ADD_GTIME": "float32", "LONGEST_ADD_GTIME": "float32",
    "DIV_ACTUAL_ELAPSED_TIME": "float32", "DIV_ARR_DELAY": "float32", "DIV_DISTANCE": "float32",
}
STR_COLS = {
    "OP_UNIQUE_CARRIER", "OP_CARRIER", "TAIL_NUM", "ORIGIN", "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_ABR", "ORIGIN_STATE_NM", "DEST", "DEST_CITY_NAME",
    "DEST_STATE_ABR", "DEST_STATE_NM", "DEP_TIME_BLK", "ARR_TIME_BLK", "CANCELLATION_CODE",
}

TRACK_A_FEATURES = [
    "YEAR", "DAY_OF_MONTH", "DAY_OF_WEEK", "IS_WEEKEND",
    "CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN",
    "CRS_DEP_SIN", "CRS_DEP_COS", "CRS_ARR_SIN", "CRS_ARR_COS",
    "CRS_ELAPSED_TIME", "DISTANCE", "DISTANCE_GROUP",
    "OP_CARRIER_FREQ", "CARRIER_HIST_OTP",
    "ORIGIN_FREQ", "ORIGIN_HIST_OTP", "DEST_FREQ",
    "ROUTE_FREQ", "DEP_TIME_BLK_FREQ",
]
TRACK_B_EXTRA = ["DEP_DELAY", "DEP_DELAY_NEW", "DEP_DEL15", "TAXI_OUT"]
TRACK_A_FORBIDDEN = {"DEP_TIME", "DEP_DELAY", "DEP_DELAY_NEW", "DEP_DEL15", "DEP_DELAY_GROUP", "TAXI_OUT", "WHEELS_OFF", "WHEELS_ON", "TAXI_IN", "ARR_TIME", "ARR_DELAY", "ARR_DELAY_NEW", "ARR_DEL15", "ARR_DELAY_GROUP", "ACTUAL_ELAPSED_TIME", "AIR_TIME", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY", "FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME", "CANCELLED", "DIVERTED"}
TRACK_B_FORBIDDEN = {"ARR_TIME", "ARR_DELAY", "ARR_DELAY_NEW", "ARR_DEL15", "ARR_DELAY_GROUP", "ARR_TIME_BLK", "ACTUAL_ELAPSED_TIME", "AIR_TIME", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY", "WHEELS_ON", "TAXI_IN", "FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME", "CANCELLED", "DIVERTED"}
DERIVED_COLUMNS = {"FL_DATE", "IS_WEEKEND", "ROUTE", "ARR_DELAY_CAT", "DOMINANT_DELAY_CAUSE", "CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN", "DEP_TIME_MIN", "ARR_TIME_MIN", "WHEELS_OFF_MIN", "WHEELS_ON_MIN", "FIRST_DEP_TIME_MIN", "CRS_DEP_SIN", "CRS_DEP_COS", "CRS_ARR_SIN", "CRS_ARR_COS"}


In [ ]:
# Utility functions copied from utils.py, transformations.py, and ml_preparation.py

def hash_inputs(paths):
    h = hashlib.md5()
    for p in sorted(paths):
        h.update(str(p).encode())
    return h.hexdigest()


def hhmm_to_minutes(s):
    v = pd.to_numeric(s, errors="coerce")
    hh = (v // 100).astype("Int32")
    mm = (v % 100).astype("Int32")
    result = hh * 60 + mm
    result = result.where((result >= 0) & (result <= 1439), other=pd.NA)
    return result


def cyclic_encode(minutes_col, period=1440):
    rad = 2 * np.pi * minutes_col / period
    return np.sin(rad).astype("float32"), np.cos(rad).astype("float32")


def standardize_columns(df):
    df.columns = df.columns.str.strip().str.upper()
    df = df[[c for c in df.columns if not c.startswith("UNNAMED")]]
    return df


def cast_dtypes(df):
    for c, dt in DTYPE_MAP_INT.items():
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(dt)
    for c, dt in DTYPE_MAP_FLOAT.items():
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(dt)
    for c in STR_COLS:
        if c in df.columns:
            df[c] = df[c].astype("string")
    return df


def apply_r0_duplicates(df, duplicate_stats, rule_counts=None):
    full_removed = int(df.duplicated().sum())
    if full_removed:
        df = df.drop_duplicates().copy()
    available_key_cols = [c for c in DUPLICATE_KEY_COLS if c in df.columns]
    key_removed = 0
    if available_key_cols:
        key_removed = int(df.duplicated(subset=available_key_cols).sum())
        if key_removed:
            df = df.drop_duplicates(subset=available_key_cols, keep="first").copy()
    duplicate_stats.setdefault("full_row_duplicates_removed", 0)
    duplicate_stats.setdefault("business_key_duplicates_removed", 0)
    duplicate_stats.setdefault("rows_removed_total", 0)
    duplicate_stats["full_row_duplicates_removed"] += full_removed
    duplicate_stats["business_key_duplicates_removed"] += key_removed
    duplicate_stats["rows_removed_total"] += full_removed + key_removed
    if rule_counts is not None:
        rule_counts["R0"] = rule_counts.get("R0", 0) + full_removed + key_removed
    return df


def apply_r1_cancelled(df):
    mask = df["CANCELLED"] == 1
    n = int(mask.sum())
    div_cols = [c for c in df.columns if c.startswith("DIV_")]
    for c in CANCEL_NULL_COLS + div_cols:
        if c in df.columns:
            df.loc[mask, c] = pd.NA if df[c].dtype.name.startswith(("Int", "string")) else np.nan
    return df, n


def apply_r2_diverted(df):
    mask = df["DIVERTED"] == 1
    n = int(mask.sum())
    for c in DIVERT_NULL_COLS:
        if c in df.columns:
            df.loc[mask, c] = pd.NA if df[c].dtype.name.startswith(("Int", "string")) else np.nan
    return df, n


def apply_r3_hhmm(df, stats):
    for c in HHMM_COLS:
        if c not in df.columns:
            continue
        mc = c + "_MIN"
        df[mc] = hhmm_to_minutes(df[c])
        parsed = int(df[mc].notna().sum())
        na = int(df[mc].isna().sum())
        stats.setdefault(c, {"parsed": 0, "na": 0})
        stats[c]["parsed"] += parsed
        stats[c]["na"] += na
        df[mc] = df[mc].astype("Int32")
    if "CRS_DEP_TIME_MIN" in df.columns:
        s, co = cyclic_encode(df["CRS_DEP_TIME_MIN"].astype("float32"))
        df["CRS_DEP_SIN"] = s
        df["CRS_DEP_COS"] = co
    if "CRS_ARR_TIME_MIN" in df.columns:
        s, co = cyclic_encode(df["CRS_ARR_TIME_MIN"].astype("float32"))
        df["CRS_ARR_SIN"] = s
        df["CRS_ARR_COS"] = co
    return df


def apply_r4_date(df, dow_mismatches):
    df["FL_DATE"] = pd.to_datetime(df["YEAR"].astype(str) + "-01-" + df["DAY_OF_MONTH"].astype(str), errors="coerce")
    computed_dow = df["FL_DATE"].dt.dayofweek.astype("Int16")
    bts_dow = df["DAY_OF_WEEK"] - 1
    mismatch = (bts_dow != computed_dow) & bts_dow.notna() & computed_dow.notna()
    dow_mismatches["count"] = dow_mismatches.get("count", 0) + int(mismatch.sum())
    df["DAY_OF_WEEK"] = computed_dow
    return df


def apply_r5_weekend(df):
    df["IS_WEEKEND"] = (df["DAY_OF_WEEK"].isin({5, 6})).astype("Int16")
    return df


def apply_r6_route(df):
    df["ROUTE"] = df["ORIGIN"].astype(str) + "-" + df["DEST"].astype(str)
    return df


def apply_r7_delay_cat(df, target_col):
    src = "ARR_DELAY" if target_col == "ARR_DEL15" else "ARR_DELAY_NEW"
    if src in df.columns:
        bins = [-np.inf, 0, 15, 60, 180, np.inf]
        labels = ["<=0", "1-15", "16-60", "61-180", ">180"]
        df["ARR_DELAY_CAT"] = pd.cut(df[src].astype("float64"), bins=bins, labels=labels)
    return df


def apply_r8_dominant(df):
    avail = [c for c in DELAY_CAUSE_COLS if c in df.columns]
    if not avail:
        return df
    sub = df[avail].astype("float32")
    has_any = sub.notna().any(axis=1) & (sub.fillna(0).sum(axis=1) > 0)
    dom = sub.idxmax(axis=1).where(has_any, other=pd.NA)
    df["DOMINANT_DELAY_CAUSE"] = dom.astype("string")
    return df


def clean_chunk(df, target_col, hhmm_stats, dow_mismatches, rule_counts, duplicate_stats):
    df = apply_r0_duplicates(df, duplicate_stats, rule_counts)
    df, n1 = apply_r1_cancelled(df)
    rule_counts["R1"] = rule_counts.get("R1", 0) + n1
    df, n2 = apply_r2_diverted(df)
    rule_counts["R2"] = rule_counts.get("R2", 0) + n2
    apply_r3_hhmm(df, hhmm_stats)
    rule_counts["R3"] = rule_counts.get("R3", 0) + len(df)
    apply_r4_date(df, dow_mismatches)
    rule_counts["R4"] = rule_counts.get("R4", 0) + len(df)
    apply_r5_weekend(df)
    apply_r6_route(df)
    apply_r7_delay_cat(df, target_col)
    apply_r8_dominant(df)
    return df


def apply_freq_otp(df, freq_maps, otp_maps, global_otp, unseen_log):
    col_map = {"OP_CARRIER": "OP_CARRIER_FREQ", "ORIGIN": "ORIGIN_FREQ", "DEST": "DEST_FREQ", "ROUTE": "ROUTE_FREQ", "DEP_TIME_BLK": "DEP_TIME_BLK_FREQ"}
    for src, dst in col_map.items():
        if src in df.columns and src in freq_maps:
            mapped = df[src].map(freq_maps[src])
            unseen = mapped.isna() & df[src].notna()
            unseen_log.setdefault(src, {"total": 0, "unseen": 0})
            unseen_log[src]["total"] += int(df[src].notna().sum())
            unseen_log[src]["unseen"] += int(unseen.sum())
            df[dst] = mapped.fillna(0).astype("float32")
    otp_col_map = {"ORIGIN": "ORIGIN_HIST_OTP", "OP_CARRIER": "CARRIER_HIST_OTP"}
    for src, dst in otp_col_map.items():
        if src in df.columns and src in otp_maps:
            mapped = df[src].map(otp_maps[src])
            unseen = mapped.isna() & df[src].notna()
            unseen_log.setdefault(f"otp_{src}", {"total": 0, "unseen": 0})
            unseen_log[f"otp_{src}"]["total"] += int(df[src].notna().sum())
            unseen_log[f"otp_{src}"]["unseen"] += int(unseen.sum())
            df[dst] = mapped.fillna(global_otp).astype("float32")
    return df


def build_ml_rows(df, target_col, freq_maps, otp_maps, global_otp, unseen_log):
    df = apply_freq_otp(df.copy(), freq_maps, otp_maps, global_otp, unseen_log)
    df = df[df[target_col].notna()].copy()
    if df.empty:
        return None, None
    a_cols = [c for c in TRACK_A_FEATURES if c in df.columns] + [target_col]
    track_a = df[a_cols].copy()
    b_cols = [c for c in TRACK_A_FEATURES + TRACK_B_EXTRA if c in df.columns] + [target_col]
    track_b = df[b_cols].copy()
    return track_a, track_b


In [ ]:
# Core pipeline and summary functions copied from pipeline.py and reporting.py

def load_mappings(mappings_dir):
    freq_maps = {}
    for col in FREQ_ENCODE_COLS:
        p = mappings_dir / f"freq_{col}.parquet"
        if p.exists():
            tmp = pd.read_parquet(p)
            freq_maps[col] = dict(zip(tmp["key"], tmp["freq"]))
    otp_maps = {}
    for col in OTP_GROUP_COLS:
        p = mappings_dir / f"otp_{col}.parquet"
        if p.exists():
            tmp = pd.read_parquet(p)
            otp_maps[col] = dict(zip(tmp["key"], tmp["otp"]))
    gp = mappings_dir / "global_otp.json"
    global_otp = json.loads(gp.read_text())["global_otp"] if gp.exists() else 0.5
    return freq_maps, otp_maps, global_otp


def run_pass1(input_files, chunksize, target_col, out_dir):
    mappings_dir = Path(out_dir) / "mappings"
    mappings_dir.mkdir(parents=True, exist_ok=True)
    meta_path = mappings_dir / "_meta.json"
    inp_hash = hash_inputs(input_files)
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        if meta.get("completed_pass1") and meta.get("inputs_hash") == inp_hash and meta.get("chunksize") == chunksize and meta.get("target") == target_col:
            log.info("Pass 1 checkpoint found and valid; skipping.")
            return load_mappings(mappings_dir)
        log.info("Pass 1 checkpoint stale; re-running.")
    log.info("=== PASS 1: Building train-only mappings (years 2021-2024) ===")
    freq_counts = {c: {} for c in FREQ_ENCODE_COLS}
    otp_counts = {c: {} for c in OTP_GROUP_COLS}
    total_train_operated = 0
    train_files = [f for f in input_files if any(str(y) in str(f) for y in TRAIN_YEARS)]
    for fpath in train_files:
        log.info(f"  Pass1 reading {fpath}")
        for ci, chunk in enumerate(pd.read_csv(fpath, chunksize=chunksize, low_memory=False)):
            chunk = standardize_columns(chunk)
            chunk = cast_dtypes(chunk)
            chunk = apply_r0_duplicates(chunk, {}, None)
            chunk, _ = apply_r1_cancelled(chunk)
            chunk, _ = apply_r2_diverted(chunk)
            apply_r3_hhmm(chunk, {})
            apply_r4_date(chunk, {})
            apply_r6_route(chunk)
            operated = chunk[(chunk["CANCELLED"] == 0) & (chunk["DIVERTED"] == 0)]
            total_train_operated += len(operated)
            for col in FREQ_ENCODE_COLS:
                if col in chunk.columns:
                    vc = chunk[col].value_counts()
                    for k, v in vc.items():
                        freq_counts[col][k] = freq_counts[col].get(k, 0) + int(v)
            for col in OTP_GROUP_COLS:
                if col in operated.columns and target_col in operated.columns:
                    grp = operated.groupby(col)[target_col].agg(["count", "sum"])
                    for k, row in grp.iterrows():
                        prev = otp_counts[col].get(k, [0, 0])
                        otp_counts[col][k] = [prev[0] + int(row["count"]), prev[1] + int(row["sum"])]
            if (ci + 1) % 3 == 0:
                log.info(f"    chunk {ci+1} done")
    freq_maps = {}
    for col, counts in freq_counts.items():
        total = sum(counts.values()) or 1
        freq_maps[col] = {k: v / total for k, v in counts.items()}
    otp_maps = {}
    global_total = 0
    global_ontime_sum = 0
    for col, mapping in otp_counts.items():
        otp_maps[col] = {}
        for k, (cnt, delayed) in mapping.items():
            otp_maps[col][k] = (cnt - delayed) / cnt if cnt > 0 else 0.5
            global_total += cnt
            global_ontime_sum += (cnt - delayed)
    global_otp = global_ontime_sum / global_total if global_total > 0 else 0.5
    global_otp = ((total_train_operated - sum(v[1] for v in otp_counts["ORIGIN"].values())) / total_train_operated if total_train_operated > 0 else 0.5)
    for col, m in freq_maps.items():
        pd.DataFrame(list(m.items()), columns=["key", "freq"]).to_parquet(mappings_dir / f"freq_{col}.parquet", index=False)
    for col, m in otp_maps.items():
        pd.DataFrame(list(m.items()), columns=["key", "otp"]).to_parquet(mappings_dir / f"otp_{col}.parquet", index=False)
    (mappings_dir / "global_otp.json").write_text(json.dumps({"global_otp": global_otp}))
    meta = {"completed_pass1": True, "inputs_hash": inp_hash, "chunksize": chunksize, "target": target_col, "total_train_operated": total_train_operated}
    meta_path.write_text(json.dumps(meta, indent=2))
    log.info(f"Pass 1 complete. Train operated rows: {total_train_operated:,}")
    return freq_maps, otp_maps, global_otp


def write_parquet_partition(df, base_dir, year_val, writers_state):
    part_dir = Path(base_dir) / f"YEAR={year_val}"
    part_dir.mkdir(parents=True, exist_ok=True)
    fpath = part_dir / "part-0.parquet"
    df_write = df.drop(columns=["YEAR"], errors="ignore")
    table = pa.Table.from_pandas(df_write, preserve_index=False)
    key = str(fpath)
    if key not in writers_state:
        writers_state[key] = pq.ParquetWriter(str(fpath), table.schema, compression="snappy")
    try:
        writers_state[key].write_table(table)
    except (pa.ArrowInvalid, pa.ArrowTypeError):
        writers_state[key].close()
        writers_state[key] = pq.ParquetWriter(str(fpath), table.schema, compression="snappy")
        writers_state[key].write_table(table)


def run_pass2(input_files, chunksize, target_col, out_dir, overwrite, freq_maps, otp_maps, global_otp):
    log.info("=== PASS 2: Full cleaning and output generation ===")
    clean_full_dir = Path(out_dir) / "clean_full"
    clean_op_dir = Path(out_dir) / "clean_operated"
    ml_a_dir = Path(out_dir) / "ml_track_a"
    ml_b_dir = Path(out_dir) / "ml_track_b"
    for d in [clean_full_dir, clean_op_dir, ml_a_dir, ml_b_dir]:
        d.mkdir(parents=True, exist_ok=True)
    if overwrite:
        for d in [clean_full_dir, clean_op_dir]:
            for sub in d.glob("YEAR=*"):
                shutil.rmtree(sub, ignore_errors=True)
    hhmm_stats = {}
    dow_mismatches = {"count": 0}
    rule_counts = {}
    duplicate_stats = {"full_row_duplicates_removed": 0, "business_key_duplicates_removed": 0, "rows_removed_total": 0}
    year_stats = {}
    writers_full = {}
    writers_op = {}
    unseen_log = {}
    ml_a_train, ml_a_test, ml_b_train, ml_b_test = [], [], [], []
    consistency = {"distance_neg": 0, "time_min_oor": 0}
    for fpath in input_files:
        log.info(f"  Pass2 reading {fpath}")
        for ci, chunk in enumerate(pd.read_csv(fpath, chunksize=chunksize, low_memory=False)):
            chunk = standardize_columns(chunk)
            chunk = cast_dtypes(chunk)
            if "YEAR" not in chunk.columns or chunk["YEAR"].isna().all():
                log.warning(f"  chunk {ci} has no YEAR column, skipping")
                continue
            years_in_chunk = chunk["YEAR"].dropna().unique()
            chunk = clean_chunk(chunk, target_col, hhmm_stats, dow_mismatches, rule_counts, duplicate_stats)
            if "DISTANCE" in chunk.columns:
                consistency["distance_neg"] += int((chunk["DISTANCE"] < 0).sum())
            for yr in years_in_chunk:
                yr = int(yr)
                yr_chunk = chunk[chunk["YEAR"] == yr]
                st = year_stats.setdefault(yr, {"rows_read": 0, "rows_full": 0, "rows_operated": 0, "cancelled": 0, "diverted": 0})
                st["rows_read"] += len(yr_chunk)
                st["rows_full"] += len(yr_chunk)
                st["cancelled"] += int((yr_chunk["CANCELLED"] == 1).sum()) if "CANCELLED" in yr_chunk.columns else 0
                st["diverted"] += int((yr_chunk["DIVERTED"] == 1).sum()) if "DIVERTED" in yr_chunk.columns else 0
                write_parquet_partition(yr_chunk, clean_full_dir, yr, writers_full)
                op = yr_chunk[(yr_chunk["CANCELLED"] == 0) & (yr_chunk["DIVERTED"] == 0)]
                st["rows_operated"] += len(op)
                if not op.empty:
                    write_parquet_partition(op, clean_op_dir, yr, writers_op)
                    ta, tb = build_ml_rows(op, target_col, freq_maps, otp_maps, global_otp, unseen_log)
                    if ta is not None:
                        if yr in TRAIN_YEARS:
                            ml_a_train.append(ta)
                            ml_b_train.append(tb)
                        else:
                            ml_a_test.append(ta)
                            ml_b_test.append(tb)
            if (ci + 1) % 3 == 0:
                log.info(f"    chunk {ci+1} done")
    for w in list(writers_full.values()) + list(writers_op.values()):
        w.close()
    ml_counts = {}
    for name, parts, d in [("ml_track_a_train", ml_a_train, ml_a_dir), ("ml_track_a_test", ml_a_test, ml_a_dir), ("ml_track_b_train", ml_b_train, ml_b_dir), ("ml_track_b_test", ml_b_test, ml_b_dir)]:
        if parts:
            combined = pd.concat(parts, ignore_index=True)
            out_path = d / f"{name}.parquet"
            combined.to_parquet(out_path, engine="pyarrow", compression="snappy", index=False)
            ml_counts[name] = len(combined)
            log.info(f"  Wrote {name}: {len(combined):,} rows, cols={list(combined.columns)}")
        else:
            ml_counts[name] = 0
            log.warning(f"  {name} has 0 rows!")
    log.info("Pass 2 complete.")
    return {"year_stats": year_stats, "hhmm_stats": hhmm_stats, "dow_mismatches": dow_mismatches, "rule_counts": rule_counts, "duplicate_stats": duplicate_stats, "consistency": consistency, "ml_counts": ml_counts, "unseen_log": unseen_log}


In [ ]:
def sample_schema_summary(out_dir):
    sample_dir = Path(out_dir) / "clean_full"
    parts = sorted(sample_dir.rglob("*.parquet"))
    if not parts:
        return {"sample_partition": None, "columns": [], "derived_columns": [], "error": None}
    try:
        sample_df = pd.read_parquet(parts[0], engine="pyarrow")
        rows = []
        for col in sample_df.columns:
            rows.append({"column": col, "dtype": str(sample_df[col].dtype), "missing_pct": round(float(sample_df[col].isna().mean() * 100), 2)})
        derived = [c for c in sample_df.columns if c in DERIVED_COLUMNS]
        return {"sample_partition": str(parts[0]), "columns": rows, "derived_columns": derived, "error": None}
    except Exception as exc:
        return {"sample_partition": None, "columns": [], "derived_columns": [], "error": str(exc)}


def build_processing_context(input_files, chunksize, target_col, out_dir, stats, start_time, end_time):
    ys = stats["year_stats"]
    elapsed = (datetime.fromisoformat(end_time) - datetime.fromisoformat(start_time)).total_seconds()
    artifacts = [
        {"directory": f"{out_dir}/clean_full/", "description": "Full cleaned data", "partition": "YEAR=YYYY/part-0.parquet"},
        {"directory": f"{out_dir}/clean_operated/", "description": "Operated only", "partition": "YEAR=YYYY/part-0.parquet"},
        {"directory": f"{out_dir}/ml_track_a/", "description": "ML pre-flight", "partition": "ml_track_a_train.parquet, ml_track_a_test.parquet"},
        {"directory": f"{out_dir}/ml_track_b/", "description": "ML post-pushback", "partition": "ml_track_b_train.parquet, ml_track_b_test.parquet"},
        {"directory": f"{out_dir}/mappings/", "description": "Train-only freq/OTP maps", "partition": "Parquet files + global_otp.json"},
    ]
    year_rows = []
    for yr in sorted(ys.keys()):
        s = ys[yr]
        rows_read = s["rows_read"]
        year_rows.append({"YEAR": int(yr), "rows_read": int(rows_read), "rows_full": int(s["rows_full"]), "rows_operated": int(s["rows_operated"]), "cancelled": int(s["cancelled"]), "diverted": int(s["diverted"]), "cancelled_pct": round((s["cancelled"] / rows_read * 100) if rows_read else 0.0, 2), "diverted_pct": round((s["diverted"] / rows_read * 100) if rows_read else 0.0, 2)})
    return {"generated_at": datetime.now().isoformat(), "run_config": {"machine": "local (~16 GB RAM)", "python": sys.version.split()[0], "pandas": pd.__version__, "pyarrow": pa.__version__, "files": [str(f) for f in input_files], "chunksize": int(chunksize), "target": target_col, "start": start_time, "end": end_time, "runtime_seconds": round(elapsed, 1)}, "ingestion_summary": {"years": year_rows, "totals": {"rows_read": int(sum(s["rows_read"] for s in ys.values())), "rows_full": int(sum(s["rows_full"] for s in ys.values())), "rows_operated": int(sum(s["rows_operated"] for s in ys.values()))}}, "schema_summary": sample_schema_summary(out_dir), "transformation_log": {"rule_counts": {k: int(v) for k, v in stats["rule_counts"].items()}, "hhmm_stats": {k: {"parsed": int(v["parsed"]), "na": int(v["na"])} for k, v in stats["hhmm_stats"].items()}, "rules": {"R0": "Duplicate handling", "R1": "Cancelled nullification", "R2": "Diverted nullification", "R3": "HHMM parsing", "R4": "FL_DATE / DOW validation", "R5": "IS_WEEKEND", "R6": "ROUTE", "R7": "ARR_DELAY_CAT", "R8": "DOMINANT_DELAY_CAUSE", "R9": "Operated subset", "R10": "Freq/OTP encoding"}}, "consistency_checks": {"dow_mismatches": int(stats["dow_mismatches"]["count"]), "distance_neg": int(stats["consistency"]["distance_neg"]), "time_min_oor": int(stats["consistency"].get("time_min_oor", 0)), "full_row_duplicates_removed": int(stats["duplicate_stats"]["full_row_duplicates_removed"]), "business_key_duplicates_removed": int(stats["duplicate_stats"]["business_key_duplicates_removed"]), "duplicate_rows_removed_total": int(stats["duplicate_stats"]["rows_removed_total"]), "decision_day_of_week": "overwrite with computed (Mon=0..Sun=6)", "decision_distance_neg": "kept as-is (data quality flag)"}, "ml_audit": {"row_counts": {k: int(v) for k, v in stats["ml_counts"].items()}, "track_a_features": TRACK_A_FEATURES, "track_b_features": TRACK_A_FEATURES + TRACK_B_EXTRA, "track_a_forbidden": sorted(TRACK_A_FORBIDDEN), "track_b_forbidden": sorted(TRACK_B_FORBIDDEN), "unseen_rates": {k: {"total": int(v["total"]), "unseen": int(v["unseen"]), "rate_pct": round((v["unseen"] / v["total"] * 100) if v["total"] else 0.0, 3)} for k, v in stats["unseen_log"].items()}, "unseen_policy": "Unseen values mapped to 0 (freq) or global OTP mean (OTP)."}, "artifacts": artifacts, "out_dir": str(out_dir)}


def acceptance_checks(out_dir, target_col):
    errors = []
    od = Path(out_dir)
    for yr in range(2021, 2026):
        for sub in ["clean_full", "clean_operated"]:
            d = od / sub / f"YEAR={yr}"
            if not d.exists():
                errors.append(f"Missing partition: {d}")
    for fname in ["ml_track_a/ml_track_a_train.parquet", "ml_track_a/ml_track_a_test.parquet", "ml_track_b/ml_track_b_train.parquet", "ml_track_b/ml_track_b_test.parquet"]:
        p = od / fname
        if not p.exists():
            errors.append(f"Missing ML file: {p}")
    if errors:
        raise AssertionError("ACCEPTANCE FAILED:\n" + "\n".join(f"  - {e}" for e in errors))
    print("Acceptance checks passed")


In [ ]:
# Parameters
INPUT_FILES = FILES.copy()
CHUNK_SIZE = CHUNKSIZE
TARGET_COL = TARGET
OUT_DIR = REPO_ROOT / "data" / "processed"
OVERWRITE = True
summary = None
stats = None
params_df = pd.DataFrame([{"param": "INPUT_FILES", "value": INPUT_FILES}, {"param": "CHUNK_SIZE", "value": CHUNK_SIZE}, {"param": "TARGET_COL", "value": TARGET_COL}, {"param": "OUT_DIR", "value": str(OUT_DIR)}, {"param": "OVERWRITE", "value": OVERWRITE}])
display(params_df)


## 1. Validate Inputs


In [ ]:
required = {"YEAR", "DAY_OF_MONTH", "DAY_OF_WEEK", "OP_CARRIER", "ORIGIN", "DEST", "CRS_DEP_TIME", "CRS_ARR_TIME", "CANCELLED", "DIVERTED", TARGET_COL}
resolved_inputs = [REPO_ROOT / Path(p) for p in INPUT_FILES]
missing_files = [str(p) for p in resolved_inputs if not p.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing input files: {missing_files}")
sample = pd.read_csv(resolved_inputs[0], nrows=0)
sample.columns = sample.columns.str.strip().str.upper()
missing_cols = sorted(required - set(sample.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")
print("Validated files:", len(resolved_inputs))
print("First file      :", resolved_inputs[0])
print("Target column   :", TARGET_COL)


## 2. Run Preprocessing


In [ ]:
start = datetime.now().isoformat()
freq_maps, otp_maps, global_otp = run_pass1(resolved_inputs, CHUNK_SIZE, TARGET_COL, OUT_DIR)
stats = run_pass2(resolved_inputs, CHUNK_SIZE, TARGET_COL, OUT_DIR, OVERWRITE, freq_maps, otp_maps, global_otp)
end = datetime.now().isoformat()
summary = build_processing_context(resolved_inputs, CHUNK_SIZE, TARGET_COL, OUT_DIR, stats, start, end)
print("Preprocessing complete")
print("Total rows processed:", f"{sum(s['rows_read'] for s in stats['year_stats'].values()):,}")
print("Runtime seconds     :", summary["run_config"]["runtime_seconds"])


## 3. Ingestion Summary


In [ ]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
ingestion_df = pd.DataFrame(summary['ingestion_summary']['years'])
display(ingestion_df)
totals_df = pd.DataFrame([summary['ingestion_summary']['totals']])
display(totals_df)


## 4. Schema and Missingness


In [ ]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
schema_summary = summary['schema_summary']
if schema_summary['error']:
    print('Schema read error:', schema_summary['error'])
else:
    print('Sample partition:', schema_summary['sample_partition'])
    display(pd.DataFrame(schema_summary['columns']))
    print('Derived columns:', schema_summary['derived_columns'])


## 5. Transformation Log


In [ ]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
rule_counts = summary['transformation_log']['rule_counts']
rules = summary['transformation_log']['rules']
rule_rows = []
for key, desc in rules.items():
    rule_rows.append({'rule': key, 'description': desc, 'count': rule_counts.get(key)})
display(pd.DataFrame(rule_rows))
hhmm_df = pd.DataFrame([{'column': k, 'parsed': v['parsed'], 'na': v['na']} for k, v in summary['transformation_log']['hhmm_stats'].items()])
display(hhmm_df)


## 6. Consistency Checks


In [ ]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
consistency_df = pd.DataFrame(summary['consistency_checks'].items(), columns=['field', 'value'])
display(consistency_df)


## 7. ML Dataset Audit


In [ ]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
ml_summary = summary['ml_audit']
display(pd.DataFrame(ml_summary['row_counts'].items(), columns=['dataset', 'rows']))
display(pd.DataFrame(ml_summary['unseen_rates']).T.reset_index().rename(columns={'index': 'source'}))
print('Track A features:', ml_summary['track_a_features'])
print('Track B features:', ml_summary['track_b_features'])
print('Track A forbidden:', ml_summary['track_a_forbidden'])
print('Track B forbidden:', ml_summary['track_b_forbidden'])
print(ml_summary['unseen_policy'])


## 8. Artifacts Produced


In [ ]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
display(pd.DataFrame(summary['artifacts']))


## 9. Optional Direct Reads


In [ ]:
clean_paths = sorted((Path(OUT_DIR) / 'clean_full').rglob('*.parquet'))
if clean_paths:
    clean_sample_df = pd.read_parquet(clean_paths[0])
    print('Clean sample shape:', clean_sample_df.shape)
    display(clean_sample_df.head())
else:
    print('No clean_full parquet files found.')
track_a_train = Path(OUT_DIR) / 'ml_track_a' / 'ml_track_a_train.parquet'
if track_a_train.exists():
    track_a_df = pd.read_parquet(track_a_train)
    print('Track A train shape:', track_a_df.shape)
    display(track_a_df.head())
else:
    print('Track A train parquet not found.')
